# V2 Market Leading Features

## 목적

V1 데이터에 시장 수급·가격 흐름·소비자 심리 지표를 추가하여
V2용 선행지표 Feature를 생성한다.

### 주요 데이터

#### 한국부동산원 R-ONE
- 아파트 매매수급동향
- 아파트 전세수급동향
- 매매가격지수
- 전세가격지수

#### 한국은행
- 주택가격전망CSI
- 소비자심리지수(CCSI)

### 기본 원칙

- 기존 `df_final_v1.parquet`는 수정하지 않는다.
- 월별 지표는 미래정보 누출을 방지하기 위해 기본적으로 `lag1`을 사용한다.
- 원본 데이터는 `data/external/`에서 불러온다.
- 최종 결과는 `data/v2_5year_seoul/processed/`에 저장한다.

In [1]:
# ============================================================
# 프로젝트 경로 설정
# - 현재 노트북 위치와 관계없이 data 폴더가 있는 프로젝트 루트를 찾는다.
# - external: 외부기관 원본 데이터 경로
# - processed: 정제 완료 데이터 저장 경로
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "프로젝트 루트에서 data 폴더를 찾을 수 없습니다."
    )


PROJECT_DIR = find_project_root()

EXTERNAL_DIR = PROJECT_DIR / "data" / "external"

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
)

print("PROJECT_DIR:", PROJECT_DIR)
print("EXTERNAL_DIR:", EXTERNAL_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_DIR: c:\Users\uk\Apartment price
EXTERNAL_DIR: c:\Users\uk\Apartment price\data\external
PROCESSED_DIR: c:\Users\uk\Apartment price\data\v2_5year_seoul\processed


In [2]:
# ============================================================
# V2 시장 선행지표 원본 파일 경로 정의
# - 한국부동산원: 매매·전세 수급 및 가격지수
# - 한국은행: 주택가격전망CSI, CCSI
# - 파일 존재 여부를 확인해 경로 오류를 사전에 점검한다.
# ============================================================

DATA_FILES = {
    "sale_supply": (
        EXTERNAL_DIR
        / "(월) 매매수급동향_아파트.csv"
    ),

    "jeonse_supply": (
        EXTERNAL_DIR
        / "(월) 전세수급동향_아파트.csv"
    ),

    "sale_price_index": (
        EXTERNAL_DIR
        / "(월) 매매가격지수_주택종합.csv"
    ),

    "jeonse_price_index": (
        EXTERNAL_DIR
        / "(월) 전세가격지수_주택종합.csv"
    ),

    "housing_csi": (
        EXTERNAL_DIR
        / "주택가격전망CSI와 미분양주택.xlsx"
    ),

    "ccsi": (
        EXTERNAL_DIR
        / "소비자심리지수(CCSI).xlsx"
    ),
}


for name, path in DATA_FILES.items():
    print(
        f"{name:20s}",
        "✅" if path.exists() else "❌",
        path.name,
    )

sale_supply          ✅ (월) 매매수급동향_아파트.csv
jeonse_supply        ✅ (월) 전세수급동향_아파트.csv
sale_price_index     ✅ (월) 매매가격지수_주택종합.csv
jeonse_price_index   ✅ (월) 전세가격지수_주택종합.csv
housing_csi          ✅ 주택가격전망CSI와 미분양주택.xlsx
ccsi                 ✅ 소비자심리지수(CCSI).xlsx
